# Space-Grade Fault Detector — harden `top` as one macro (Classic flow)

Adapted from the chipathon-2026 `01_rtl2gds_counter.ipynb` skeleton. Differences from that notebook:
- RTL already exists in your repo (`rtl/*.v`, 11 modules) — not inlined here.
- Runs **directly** in-container via `subprocess` — no `docker exec` hop, since the kernel is already inside `gf180`.
- Uses a real `constraints.sdc` (`FALLBACK_SDC`) instead of relying on LibreLane's default `base.sdc`.
- No padring, no chip-top wrapper yet — that's the next notebook. This one only produces a hardened `top` macro: `gds/lef/lib/v`.

RUN_LIBRELANE defaults to False — flip it once Step 1's port names are confirmed.

## Step 0 — configuration

In [ ]:
from pathlib import Path
import subprocess, textwrap, csv, yaml

RUN_LIBRELANE = True   

# One filesystem, one kernel — you're already inside gf180. No host/container split.
PROJECT_DIR = Path("/foss/designs/Space-Grade-Mechanical-Fault-Detector")
PDK_ROOT     = "/foss/pdks"
PDK_NAME     = "gf180mcuD"
STD_CELL_LIB = "gf180mcu_fd_sc_mcu7t5v0"

DESIGN_NAME  = "top"
RUN_TAG      = "macro_v1"
CLOCK_PERIOD = 62.5   # ns -> 16 MHz

print("Project dir:", PROJECT_DIR)
print("Exists:", PROJECT_DIR.exists())

Project dir: /foss/designs/Space-Grade-Mechanical-Fault-Detector
Exists: True


## Step 1 — confirm CLOCK_PORT / RESET_PORT from top.v
Don't guess these into the config below. Read the actual port declaration first.

In [12]:
top_v = (PROJECT_DIR / "rtl" / "top.v").read_text()
print(top_v[:2000])

# Fill these in from the printed module port list before running Step 3:
CLOCK_PORT = "clk"
RESET_PORT = "sys_rst_n"   # used in Step 2 to patch constraints.sdc

//============================================================================
// top.v -- Space-Grade Vibration Pattern Anomaly Detector
// Integrates: spi_apb_interface (owns spi_master), axis_sequencer,
// goertzel_core, magnitude_compute, fault_flagger, tmr_reg_bank, apb
//============================================================================
`timescale 1ns/1ps
`default_nettype none



module top (
    input  wire        clk,
    input  wire        sys_rst_n,

    // IIS3DWB sensor SPI pins
    input  wire        c_miso,
    output wire        c_csn,
    output wire        c_sclk,
    output wire        c_mosi,

    // DRDY interrupt from sensor
    input  wire        sensor_drdy,

    // tmr_forward_en: 0=Option A (local read only), 1=Option B (also
    // push samples to tmr_reg_bank over apb)
    input  wire        tmr_forward_en,

    // Fault output to RISC core
    output wire        fault_flag_out
);

    // -------------------------------------------------------------

## Step 2 — confirm constraints.sdc, patch in the reset port
Upload `constraints.sdc` directly into `PROJECT_DIR` via the Jupyter file browser (drag-and-drop into the
`Space-Grade-Mechanical-Fault-Detector` folder on the left panel) — no copy-from-elsewhere step needed
since everything is already one filesystem. This cell just patches the reset line in place.

In [13]:
sdc_path = PROJECT_DIR / "constraints.sdc"

if not sdc_path.exists():
    print(f"Not found: {sdc_path} — upload it via the Jupyter file browser first, then re-run this cell.")
else:
    sdc_text = sdc_path.read_text()
    if "REPLACE" not in RESET_PORT:
        sdc_text = sdc_text.replace(
            "# set_false_path -from [get_ports RESET_PORT_NAME]",
            f"set_false_path -from [get_ports {RESET_PORT}]"
        )
        sdc_path.write_text(sdc_text)
        print(f"Patched reset false-path into {sdc_path}")
    else:
        print("RESET_PORT still a placeholder — fill it in from Step 1 before patching.")

Patched reset false-path into /foss/designs/Space-Grade-Mechanical-Fault-Detector/constraints.sdc


## Step 3 — pre-flight: confirm TMR keep-attributes are in the RTL
`(* keep *)` / `dont_touch` must already be present in `tmr_reg_bank.v`, `axis_sequencer.v`,
`goertzel_core.v`, `magnitude_compute.v` — otherwise Yosys will merge the redundant register
copies during Step 5 and TMR will silently disappear from the netlist.

In [15]:
for fname in ["tmr_reg_bank.v", "axis_sequencer.v", "goertzel_core.v", "magnitude_compute.v"]:
    text = (PROJECT_DIR / "rtl" / fname).read_text()
    has_keep = "keep" in text or "dont_touch" in text
    print(f"{fname:25s} keep/dont_touch present: {has_keep}")

tmr_reg_bank.v            keep/dont_touch present: True
axis_sequencer.v          keep/dont_touch present: True
goertzel_core.v           keep/dont_touch present: True
magnitude_compute.v       keep/dont_touch present: True


## Step 4 — run ENTIRE LibreLane (direct subprocess, no docker exec)

In [19]:
flow_script = textwrap.dedent(f"""
    set -euo pipefail
    cd {PROJECT_DIR}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    librelane config.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {PDK_ROOT} \\
        --manual-pdk \\
        --run-tag {RUN_TAG}
""").strip()

print("$", flow_script)
if RUN_LIBRELANE:
    proc = subprocess.run(["bash", "-lc", flow_script], capture_output=True, text=True, timeout=None)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-2000:])
    print("returncode:", proc.returncode)
else:
    print("(skipped — flip RUN_LIBRELANE)")

$ set -euo pipefail
cd /foss/designs/Space-Grade-Mechanical-Fault-Detector
source sak-pdk-script.sh gf180mcuD gf180mcu_fd_sc_mcu7t5v0
librelane config.yaml \
    --pdk gf180mcuD \
    --pdk-root /foss/pdks \
    --manual-pdk \
    --run-tag macro_v1
 floating nets.                                            
[11:54:02] WARNING  [OpenROAD.ResizerTimingPostCTS] [RSZ-0062]       ]8;id=6051579;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py\flow.py]8;;\:]8;id=6051580;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py#701\701]8;;\
                    Unable to repair all setup violations.                      
[11:54:02] WARNING  [Odb.PortDiodePlacement] 'GPL_CELL_PADDING' is   ]8;id=6051585;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py\flow.py]8;;\:]8;id=6051586;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py#701\701]8;;\
                    set to 0. This step may cause overlap failu

## Step 5 — read metrics
Key is `design__die__area` (no `__um2` suffix) — confirmed from the validated counter run.

In [ ]:
metrics_path = PROJECT_DIR / "runs" / RUN_TAG / "final" / "metrics.csv"
wanted = [
    "design__die__area",
    "design__instance__count__stdcell",
    "timing__setup_vio__count",
    "timing__hold_vio__count",
    "magic__drc_error__count",
    "klayout__drc_error__count",
    "design__lvs_error__count",
    "power__total",
]

found = {}
if metrics_path.exists():
    with metrics_path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in wanted:
                found[row[0]] = row[1] if len(row) > 1 else ""
    for key in wanted:
        print(f"  {key:35s} {found.get(key, '(missing)')}")
else:
    print(f"Not found: {metrics_path} — Step 5 hasn't completed yet.")

## Step 6 — confirm TMR flop count survived synthesis
Check the Yosys stats report before treating this run as done.

In [ ]:
run_dir = PROJECT_DIR / "runs" / RUN_TAG
stat_rpts = sorted(run_dir.glob("*-yosys-synthesis/reports/*.stat.rpt")) if run_dir.exists() else []
for p in stat_rpts:
    print(p)
print("Open the report above and confirm tmr_reg_bank flop count is 3x the single-copy count, not deduplicated.")

## Next
Once Step 5/6 are clean (all signoff metrics zero, TMR intact): this `top` macro's
`gds/lef/lib/v` quartet feeds the next notebook — chip-top + padring, following the
`02`/`03` pattern (`MACROS:` dict, `PDN_MACRO_CONNECTIONS:`, `SLOT=` selection).